# ST-CGAN: Spatio-Temporal Conditional GAN for Post-Fire Vegetation Recovery

This notebook trains a Spatio-Temporal Conditional GAN on MODIS MOD13Q1 NDVI/EVI 16-day composites to generate post-fire vegetation recovery maps.

## Sections
1. [Setup & Dependencies](#1.-Setup-&-Dependencies)
2. [Data Loading & Preprocessing](#2.-Data-Loading-&-Preprocessing)
3. [Model Architecture](#3.-Model-Architecture)
4. [Training](#4.-Training)
5. [Baseline Comparison](#5.-Baseline-Comparison)
6. [Evaluation & Visualization](#6.-Evaluation-&-Visualization)
7. [Uncertainty Estimation](#7.-Uncertainty-Estimation)

---
## 1. Setup & Dependencies

In [ ]:
# Install required packages (Kaggle-available)
!pip install -q rasterio==1.3.9 scikit-image==0.21.0 scipy==1.11.4 tqdm==4.66.1

In [ ]:
import os, sys, glob, re, warnings, csv, time
import random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # no external display server needed
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from tqdm import tqdm

import rasterio
from scipy.ndimage import generic_filter
from skimage.metrics import structural_similarity as ssim_fn

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

warnings.filterwarnings('ignore')

# ── Global Seed (Req 7.3) ────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Device (Req 7.7) ─────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f'✓ GPU detected: {torch.cuda.get_device_name(0)}')
else:
    DEVICE = torch.device('cpu')
    print('⚠ WARNING: No GPU found — training will be significantly slower on CPU.')

# ── Output directory (Req 7.5) ───────────────────────────────────────────────
OUT_DIR = Path('/kaggle/working')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output directory: {OUT_DIR}')

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
CFG = dict(
    # Data
    input_dir     = '/kaggle/input',
    patch_size    = 64,
    stride        = 32,
    seq_len_in    = 6,          # conditioning frames
    seq_len_out   = 1,          # predicted frames
    train_frac    = 0.70,
    val_frac      = 0.15,
    # Architecture
    channels      = 2,          # NDVI + EVI
    ngf           = 64,
    ndf           = 64,
    dropout_rate  = 0.3,
    patch_n       = 70,         # PatchGAN patch size
    # Training
    epochs        = 200,
    batch_size    = 8,
    lr            = 2e-4,
    beta1         = 0.5,
    beta2         = 0.999,
    lambda_l1     = 100,
    patience      = 30,
    lr_factor     = 0.5,
    ckpt_every    = 25,
    # Uncertainty
    mc_samples    = 30,
)
print('Hyperparameters loaded.')

---
## 2. Data Loading & Preprocessing

In [ ]:
# ── 2.1 Discover MOD13Q1 raster files (Req 1.1) ───────────────────────────────
def discover_modis_files(input_dir: str):
    """Find all NDVI and EVI GeoTIFF files under input_dir."""
    patterns = ['**/*NDVI*.tif', '**/*EVI*.tif',
                '**/*ndvi*.tif', '**/*evi*.tif',
                '**/*.tif']
    found = set()
    for p in patterns:
        found.update(glob.glob(os.path.join(input_dir, p), recursive=True))
    files = sorted(found)
    print(f'Discovered {len(files)} raster file(s) under {input_dir}')
    return files

all_files = discover_modis_files(CFG['input_dir'])

In [ ]:
# ── 2.2 Parse temporal order & load arrays (Req 1.2) ─────────────────────────
def extract_date_from_filename(fp: str):
    """Extract YYYY-DDD or YYYYMMDD from filename; return sortable string."""
    bn = os.path.basename(fp)
    # MOD13Q1 naming: MOD13Q1.AYYYYDDD.*
    m = re.search(r'A(\d{7})', bn)   # AYYYYDDD
    if m:
        return m.group(1)
    m = re.search(r'(\d{8})', bn)    # YYYYMMDD fallback
    if m:
        return m.group(1)
    return bn                         # last resort: filename sort


def load_raster(fp: str, fill_val_threshold: float = -3000.0):
    """Load a single-band raster and return float32 array with NaN for fill."""
    with rasterio.open(fp) as src:
        arr = src.read(1).astype(np.float32)
        nodata = src.nodata
    if nodata is not None:
        arr[arr == nodata] = np.nan
    arr[arr < fill_val_threshold] = np.nan   # MODIS fill sentinel
    return arr


def neighbour_fill(arr: np.ndarray, radius: int = 1):
    """Replace NaN with spatial mean of valid neighbours (Req 1.3)."""
    def _fill(window):
        centre = window[len(window) // 2]
        if np.isnan(centre):
            valid = window[~np.isnan(window)]
            return np.nanmean(valid) if len(valid) > 0 else 0.0
        return centre
    size = 2 * radius + 1
    filled = generic_filter(arr, _fill, size=size, mode='nearest')
    return filled.astype(np.float32)


def load_dataset(files):
    """
    Load and spatially align all rasters.
    Returns array of shape (T, H, W, 2): channel 0=NDVI, channel 1=EVI.
    Assumes files come in NDVI/EVI pairs OR single-channel files (auto-assigned).
    """
    if len(files) == 0:
        print('⚠ No files found — generating synthetic data for demonstration.')
        return _synthetic_dataset()

    # Sort by date
    files_sorted = sorted(files, key=extract_date_from_filename)

    # Try to split into NDVI / EVI
    ndvi_files = [f for f in files_sorted if 'ndvi' in f.lower() or 'NDVI' in f]
    evi_files  = [f for f in files_sorted if 'evi'  in f.lower() or 'EVI'  in f]

    if len(ndvi_files) == 0 or len(evi_files) == 0:
        # Treat all files as sequential single channel; duplicate for 2 channels
        ndvi_files = files_sorted
        evi_files  = files_sorted

    T = min(len(ndvi_files), len(evi_files))
    print(f'Loading {T} temporal composites …')

    # Determine output spatial shape from first file
    ref = load_raster(ndvi_files[0])
    H, W = ref.shape

    data = np.zeros((T, H, W, 2), dtype=np.float32)

    for t in tqdm(range(T), desc='Loading composites'):
        ndvi = load_raster(ndvi_files[t])
        evi  = load_raster(evi_files[t])
        # Resize if shapes differ
        if ndvi.shape != (H, W):
            from skimage.transform import resize
            ndvi = resize(ndvi, (H, W), anti_aliasing=True)
        if evi.shape != (H, W):
            from skimage.transform import resize
            evi  = resize(evi,  (H, W), anti_aliasing=True)
        data[t, :, :, 0] = neighbour_fill(ndvi)
        data[t, :, :, 1] = neighbour_fill(evi)

    return data


def _synthetic_dataset(T=200, H=128, W=128, C=2):
    """Synthetic sinusoidal NDVI/EVI for smoke-testing when no real data exist."""
    print('Generating synthetic dataset (T=%d, H=%d, W=%d, C=%d)' % (T, H, W, C))
    t_idx = np.linspace(0, 4 * np.pi, T)
    base  = 0.5 + 0.3 * np.sin(t_idx)[:, None, None]  # (T,1,1)
    noise = 0.05 * np.random.randn(T, H, W)
    ndvi  = np.clip(base + noise, 0, 1).astype(np.float32)
    evi   = np.clip(base * 0.85 + 0.05 * np.random.randn(T, H, W), 0, 1).astype(np.float32)
    data  = np.stack([ndvi, evi], axis=-1)  # (T,H,W,2)
    return data


raw_data = load_dataset(all_files)
print(f'Raw data shape: {raw_data.shape}  (T, H, W, C)')

In [ ]:
# ── 2.3 Temporal train/val/test split (Req 1.5) ──────────────────────────────
T = raw_data.shape[0]
n_train = int(T * CFG['train_frac'])
n_val   = int(T * CFG['val_frac'])
n_test  = T - n_train - n_val

train_raw = raw_data[:n_train]
val_raw   = raw_data[n_train : n_train + n_val]
test_raw  = raw_data[n_train + n_val :]

print(f'Split → train: {n_train}  val: {n_val}  test: {n_test} composites')

# ── 2.4 Per-channel min-max normalisation on train split only (Req 1.4) ──────
ch_min = train_raw.reshape(-1, 2).min(axis=0)   # (C,)
ch_max = train_raw.reshape(-1, 2).max(axis=0)
ch_range = np.where(ch_max - ch_min > 0, ch_max - ch_min, 1.0)

def normalise(arr):
    return (arr - ch_min) / ch_range

def denormalise(arr):
    return arr * ch_range + ch_min

train_norm = normalise(train_raw)
val_norm   = normalise(val_raw)
test_norm  = normalise(test_raw)

# Round-trip verification (Req 1.7)
reconstructed = denormalise(train_norm)
mae_roundtrip = np.mean(np.abs(reconstructed - train_raw))
assert mae_roundtrip < 1e-5, f'Round-trip MAE {mae_roundtrip:.2e} exceeds 1e-5'
print(f'✓ Round-trip MAE = {mae_roundtrip:.2e}  (< 1e-5)')

In [ ]:
# ── 2.5 Patch extraction (Req 1.6) ───────────────────────────────────────────
def extract_patches(data, patch_size, stride, seq_in, seq_out):
    """
    data: (T, H, W, C)
    Returns list of (input_seq, target_seq) arrays:
        input_seq : (seq_in,  patch, patch, C)
        target_seq: (seq_out, patch, patch, C)
    """
    T, H, W, C = data.shape
    P, S = patch_size, stride
    samples = []

    for t in range(T - seq_in - seq_out + 1):
        x_seq = data[t          : t + seq_in ]
        y_seq = data[t + seq_in : t + seq_in + seq_out]
        for r in range(0, H - P + 1, S):
            for c in range(0, W - P + 1, S):
                x_patch = x_seq[:, r:r+P, c:c+P, :]  # (seq_in, P, P, C)
                y_patch = y_seq[:, r:r+P, c:c+P, :]  # (seq_out, P, P, C)
                samples.append((x_patch, y_patch))
    return samples

P  = CFG['patch_size']
S  = CFG['stride']
SI = CFG['seq_len_in']
SO = CFG['seq_len_out']

train_samples = extract_patches(train_norm, P, S, SI, SO)
val_samples   = extract_patches(val_norm,   P, S, SI, SO)
test_samples  = extract_patches(test_norm,  P, S, SI, SO)

print(f'Patches → train: {len(train_samples)}  val: {len(val_samples)}  test: {len(test_samples)}')


# ── 2.6 PyTorch Dataset ───────────────────────────────────────────────────────
class VegetationDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        # x: (SI, P, P, C) → (SI*C, P, P)  [flatten time+channel for Generator]
        x_t = torch.from_numpy(x).permute(0, 3, 1, 2).reshape(-1, P, P)  # (SI*C, P, P)
        y_t = torch.from_numpy(y).permute(0, 3, 1, 2).reshape(-1, P, P)  # (SO*C, P, P)
        return x_t.float(), y_t.float()


def make_loader(samples, batch_size, shuffle):
    ds = VegetationDataset(samples)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, pin_memory=torch.cuda.is_available())

train_loader = make_loader(train_samples, CFG['batch_size'], shuffle=True)
val_loader   = make_loader(val_samples,   CFG['batch_size'], shuffle=False)
test_loader  = make_loader(test_samples,  CFG['batch_size'], shuffle=False)

IN_CH  = SI * CFG['channels']   # Generator input channels
OUT_CH = SO * CFG['channels']   # Generator output channels
print(f'Input channels: {IN_CH}   Output channels: {OUT_CH}')

---
## 3. Model Architecture

In [ ]:
# ── 3.1 Utility blocks ────────────────────────────────────────────────────────
class ConvBNReLU(nn.Module):
    def __init__(self, in_c, out_c, k=4, s=2, p=1, use_bn=True, act='leaky'):
        super().__init__()
        layers = [nn.Conv2d(in_c, out_c, k, s, p, bias=not use_bn)]
        if use_bn:
            layers.append(nn.BatchNorm2d(out_c))
        if act == 'leaky':
            layers.append(nn.LeakyReLU(0.2, inplace=True))
        elif act == 'relu':
            layers.append(nn.ReLU(inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class ConvTransposeBNReLU(nn.Module):
    def __init__(self, in_c, out_c, k=4, s=2, p=1, dropout=0.0):
        super().__init__()
        layers = [
            nn.ConvTranspose2d(in_c, out_c, k, s, p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

In [ ]:
# ── 3.2 ConvLSTM cell ─────────────────────────────────────────────────────────
class ConvLSTMCell(nn.Module):
    def __init__(self, in_c, hidden_c, k=3):
        super().__init__()
        p = k // 2
        self.hidden_c = hidden_c
        self.gates = nn.Conv2d(in_c + hidden_c, 4 * hidden_c, k, 1, p, bias=True)

    def forward(self, x, h, c):
        combined = torch.cat([x, h], dim=1)
        gates = self.gates(combined)
        i, f, g, o = gates.chunk(4, dim=1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c_next = f * c + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

    def init_hidden(self, B, H, W, device):
        hc = self.hidden_c
        return (torch.zeros(B, hc, H, W, device=device),
                torch.zeros(B, hc, H, W, device=device))

In [ ]:
# ── 3.3 U-Net Generator with ConvLSTM bottleneck (Req 2.1, 2.2, 2.6) ─────────
class UNetGenerator(nn.Module):
    """
    U-Net encoder-decoder with:
      - Skip connections preserving spatial detail.
      - ConvLSTM in the bottleneck for temporal modelling.
      - Dropout in decoder for MC uncertainty estimation.
    Input : (B, IN_CH, P, P)
    Output: (B, OUT_CH, P, P)
    """
    def __init__(self, in_ch, out_ch, ngf=64, dropout=0.3):
        super().__init__()
        self.dropout = dropout

        # ── Encoder ──────────────────────────────────────────────────────────
        self.e1 = nn.Conv2d(in_ch, ngf, 4, 2, 1, bias=False)        # P/2
        self.e2 = ConvBNReLU(ngf,     ngf*2,  act='leaky')           # P/4
        self.e3 = ConvBNReLU(ngf*2,   ngf*4,  act='leaky')           # P/8
        self.e4 = ConvBNReLU(ngf*4,   ngf*8,  act='leaky')           # P/16

        # ── Bottleneck ConvLSTM ───────────────────────────────────────────────
        self.clstm = ConvLSTMCell(ngf*8, ngf*8)

        # ── Decoder (with skip connections) ──────────────────────────────────
        self.d1 = ConvTransposeBNReLU(ngf*8,   ngf*4, dropout=dropout)  # P/8
        self.d2 = ConvTransposeBNReLU(ngf*8,   ngf*2, dropout=dropout)  # P/4  (skip from e3)
        self.d3 = ConvTransposeBNReLU(ngf*4,   ngf,   dropout=dropout)  # P/2  (skip from e2)
        self.d4 = nn.Sequential(
            nn.ConvTranspose2d(ngf*2, out_ch, 4, 2, 1),                  # P    (skip from e1)
            nn.Tanh()
        )

    def forward(self, x):
        # x: (B, IN_CH, P, P) — treat as a single "frame" for the LSTM
        B, _, H4, W4 = x.shape

        # Encoder
        s1 = self.e1(x)                          # (B, ngf,   P/2,  P/2)
        s2 = self.e2(F.leaky_relu(s1, 0.2))      # (B, ngf*2, P/4,  P/4)
        s3 = self.e3(s2)                          # (B, ngf*4, P/8,  P/8)
        s4 = self.e4(s3)                          # (B, ngf*8, P/16, P/16)

        # Bottleneck ConvLSTM (single-step)
        _, BH, BW = s4.shape[1], s4.shape[2], s4.shape[3]
        h, c = self.clstm.init_hidden(B, BH, BW, s4.device)
        h, c = self.clstm(s4, h, c)              # temporal state

        # Decoder
        d1 = self.d1(h)                           # (B, ngf*4, P/8,  P/8)
        d2 = self.d2(torch.cat([d1, s3], dim=1))  # (B, ngf*2, P/4,  P/4)
        d3 = self.d3(torch.cat([d2, s2], dim=1))  # (B, ngf,   P/2,  P/2)
        out = self.d4(torch.cat([d3, s1], dim=1)) # (B, out_ch, P,    P)
        return out


netG = UNetGenerator(IN_CH, OUT_CH, ngf=CFG['ngf'], dropout=CFG['dropout_rate']).to(DEVICE)
print('Generator instantiated.')

In [ ]:
# ── 3.4 PatchGAN Discriminator with Spectral Norm (Req 2.3, 2.4, 2.5) ────────
class PatchGANDiscriminator(nn.Module):
    """
    Input : concatenation of condition (IN_CH) and target (OUT_CH).
    Output: patch-level plausibility score map.
    All Conv layers use spectral normalisation.
    """
    def __init__(self, in_ch, out_ch, ndf=64):
        super().__init__()
        c_in = in_ch + out_ch
        SN   = nn.utils.spectral_norm

        self.net = nn.Sequential(
            SN(nn.Conv2d(c_in,    ndf,   4, 2, 1, bias=False)), nn.LeakyReLU(0.2, True),
            SN(nn.Conv2d(ndf,     ndf*2, 4, 2, 1, bias=False)), nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, True),
            SN(nn.Conv2d(ndf*2,   ndf*4, 4, 2, 1, bias=False)), nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, True),
            SN(nn.Conv2d(ndf*4,   ndf*8, 4, 1, 1, bias=False)), nn.BatchNorm2d(ndf*8), nn.LeakyReLU(0.2, True),
            SN(nn.Conv2d(ndf*8,   1,     4, 1, 1, bias=False)),  # patch logits
        )

    def forward(self, cond, target):
        x = torch.cat([cond, target], dim=1)
        return self.net(x)


netD = PatchGANDiscriminator(IN_CH, OUT_CH, ndf=CFG['ndf']).to(DEVICE)
print('Discriminator instantiated.')

In [ ]:
# ── 3.5 Model summaries (Req 2.7) ─────────────────────────────────────────────
def model_summary(model, name):
    print(f'\n{'='*60}')
    print(f'  {name}  —  Parameter count')
    print('='*60)
    total = 0
    for n, p in model.named_parameters():
        if p.requires_grad:
            total += p.numel()
            print(f'  {n:50s}  {list(p.shape)}')
    print('-'*60)
    print(f'  Total trainable parameters: {total:,}')
    print('='*60)

model_summary(netG, 'UNet Generator')
model_summary(netD, 'PatchGAN Discriminator')

---
## 4. Training

In [ ]:
# ── 4.1 Loss functions & optimisers (Req 3.2, 3.3) ───────────────────────────
criterion_GAN = nn.BCEWithLogitsLoss()
criterion_L1  = nn.L1Loss()

optG = Adam(netG.parameters(), lr=CFG['lr'], betas=(CFG['beta1'], CFG['beta2']))
optD = Adam(netD.parameters(), lr=CFG['lr'], betas=(CFG['beta1'], CFG['beta2']))

# LR scheduler on validation SSIM plateau (Req 3.5)
schedulerG = ReduceLROnPlateau(optG, mode='max', factor=CFG['lr_factor'],
                               patience=CFG['patience'], verbose=True)

# CSV logger (Req 3.7)
LOG_CSV = OUT_DIR / 'training_log.csv'
with open(LOG_CSV, 'w', newline='') as f:
    csv.writer(f).writerow(['epoch', 'loss_G', 'loss_D', 'val_ssim'])

print('Losses, optimisers, and logger ready.')

In [ ]:
# ── 4.2 SSIM helper ───────────────────────────────────────────────────────────
def batch_ssim(pred, real):
    """
    pred, real: tensors (B, C, H, W) in [-1,1] or [0,1]
    Returns mean SSIM over batch.
    """
    pred_np = pred.detach().cpu().numpy()
    real_np = real.detach().cpu().numpy()
    scores = []
    for b in range(pred_np.shape[0]):
        p = pred_np[b].transpose(1, 2, 0)  # (H,W,C)
        r = real_np[b].transpose(1, 2, 0)
        p = np.clip(p, 0, 1)
        r = np.clip(r, 0, 1)
        score = ssim_fn(p, r, data_range=1.0, channel_axis=-1)
        scores.append(score)
    return float(np.mean(scores))

In [ ]:
# ── 4.3 Training loop (Req 3.1 – 3.8) ────────────────────────────────────────
best_val_ssim  = -np.inf
train_g_losses = []
train_d_losses = []
val_ssims      = []

def save_checkpoint(epoch, tag=''):
    fn = OUT_DIR / f'checkpoint_epoch{epoch:03d}{tag}.pt'
    torch.save({
        'epoch'       : epoch,
        'G_state'     : netG.state_dict(),
        'D_state'     : netD.state_dict(),
        'optG_state'  : optG.state_dict(),
        'optD_state'  : optD.state_dict(),
        'best_val_ssim': best_val_ssim,
    }, fn)
    print(f'  ✓ Checkpoint saved: {fn.name}')


print(f'Starting training for {CFG["epochs"]} epochs …')
print('='*65)

for epoch in range(1, CFG['epochs'] + 1):
    netG.train(); netD.train()
    ep_g_loss = ep_d_loss = 0.0
    n_batches = 0

    for cond, real in train_loader:
        cond = cond.to(DEVICE)   # (B, IN_CH, P, P)
        real = real.to(DEVICE)   # (B, OUT_CH, P, P)
        B    = cond.size(0)

        # ── Update Discriminator (Req 3.4) ───────────────────────────────────
        netD.zero_grad()
        fake = netG(cond).detach()

        real_logits = netD(cond, real)
        fake_logits = netD(cond, fake)
        real_labels = torch.ones_like(real_logits)
        fake_labels = torch.zeros_like(fake_logits)

        loss_D_real = criterion_GAN(real_logits, real_labels)
        loss_D_fake = criterion_GAN(fake_logits, fake_labels)
        loss_D = (loss_D_real + loss_D_fake) * 0.5
        loss_D.backward()
        optD.step()

        # ── Update Generator (Req 3.2) ────────────────────────────────────────
        netG.zero_grad()
        fake = netG(cond)
        fake_logits = netD(cond, fake)

        loss_G_adv = criterion_GAN(fake_logits, torch.ones_like(fake_logits))
        loss_G_l1  = criterion_L1(fake, real) * CFG['lambda_l1']
        loss_G     = loss_G_adv + loss_G_l1
        loss_G.backward()
        optG.step()

        ep_g_loss += loss_G.item()
        ep_d_loss += loss_D.item()
        n_batches += 1

    avg_g = ep_g_loss / max(n_batches, 1)
    avg_d = ep_d_loss / max(n_batches, 1)

    # ── Validation SSIM ───────────────────────────────────────────────────────
    netG.eval()
    val_ssim_scores = []
    with torch.no_grad():
        for cond_v, real_v in val_loader:
            cond_v, real_v = cond_v.to(DEVICE), real_v.to(DEVICE)
            fake_v = netG(cond_v)
            val_ssim_scores.append(batch_ssim(fake_v, real_v))
    val_ssim = float(np.mean(val_ssim_scores))

    train_g_losses.append(avg_g)
    train_d_losses.append(avg_d)
    val_ssims.append(val_ssim)

    # LR scheduler step (Req 3.5)
    schedulerG.step(val_ssim)

    # CSV log (Req 3.7)
    with open(LOG_CSV, 'a', newline='') as f:
        csv.writer(f).writerow([epoch, round(avg_g, 6), round(avg_d, 6), round(val_ssim, 6)])

    # Best checkpoint (Req 3.6)
    if val_ssim > best_val_ssim:
        best_val_ssim = val_ssim
        save_checkpoint(epoch, tag='_best')

    # Periodic checkpoint every 25 epochs (Req 3.6)
    if epoch % CFG['ckpt_every'] == 0:
        save_checkpoint(epoch)

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch [{epoch:3d}/{CFG["epochs"]}]  '
              f'G: {avg_g:.4f}  D: {avg_d:.4f}  val_SSIM: {val_ssim:.4f}  '
              f'best: {best_val_ssim:.4f}')

# ── Save final weights (Req 3.8) ──────────────────────────────────────────────
torch.save(netG.state_dict(), OUT_DIR / 'generator_final.pt')
torch.save(netD.state_dict(), OUT_DIR / 'discriminator_final.pt')
print(f'\n✓ Final weights saved to {OUT_DIR}')
print(f'✓ Training complete.  Best val SSIM: {best_val_ssim:.4f}')

In [ ]:
# ── 4.4 Training curves ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
ep_range = range(1, len(train_g_losses) + 1)

axes[0].plot(ep_range, train_g_losses, label='Generator Loss',     color='steelblue')
axes[0].plot(ep_range, train_d_losses, label='Discriminator Loss', color='tomato')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Losses'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep_range, val_ssims, color='seagreen', label='Validation SSIM')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('SSIM')
axes[1].set_title('Validation SSIM'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved.')

---
## 5. Baseline Comparison

In [ ]:
# ── 5.1 ConvLSTM Baseline (Req 4.1) ──────────────────────────────────────────
class ConvLSTMBaseline(nn.Module):
    """Multi-step ConvLSTM encoder → decoder."""
    def __init__(self, in_ch, out_ch, hidden=64):
        super().__init__()
        self.encoder = nn.Conv2d(in_ch, hidden, 3, 1, 1)
        self.cell    = ConvLSTMCell(hidden, hidden)
        self.decoder = nn.Conv2d(hidden, out_ch, 3, 1, 1)

    def forward(self, x):
        B, _, H, W = x.shape
        feat = F.relu(self.encoder(x))
        h, c = self.cell.init_hidden(B, H, W, x.device)
        h, c = self.cell(feat, h, c)
        return torch.tanh(self.decoder(h))


# ── 5.2 Seq2Seq Baseline (Req 4.2) ────────────────────────────────────────────
class Seq2SeqBaseline(nn.Module):
    """Convolutional encoder-decoder (no LSTM)."""
    def __init__(self, in_ch, out_ch, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch,   hidden,   3, 1, 1), nn.ReLU(True),
            nn.Conv2d(hidden,  hidden*2, 3, 1, 1), nn.ReLU(True),
            nn.Conv2d(hidden*2, hidden,  3, 1, 1), nn.ReLU(True),
            nn.Conv2d(hidden,  out_ch,   3, 1, 1), nn.Tanh(),
        )

    def forward(self, x):
        return self.net(x)


baseline_clstm  = ConvLSTMBaseline(IN_CH, OUT_CH).to(DEVICE)
baseline_seq2seq = Seq2SeqBaseline(IN_CH, OUT_CH).to(DEVICE)
print('Baseline models instantiated.')

In [ ]:
# ── 5.3 Train baselines (Req 4.3) ─────────────────────────────────────────────
def train_baseline(model, name, epochs, loader_tr, loader_vl):
    opt = Adam(model.parameters(), lr=CFG['lr'], betas=(CFG['beta1'], CFG['beta2']))
    loss_fn = nn.L1Loss()
    print(f'\nTraining {name} for {epochs} epochs …')
    for ep in range(1, epochs + 1):
        model.train()
        for cond, real in loader_tr:
            cond, real = cond.to(DEVICE), real.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(cond), real)
            loss.backward(); opt.step()
        if ep % 50 == 0 or ep == epochs:
            print(f'  Epoch {ep}/{epochs}  L1: {loss.item():.4f}')
    return model

baseline_clstm   = train_baseline(baseline_clstm,   'ConvLSTM Baseline',  CFG['epochs'], train_loader, val_loader)
baseline_seq2seq = train_baseline(baseline_seq2seq,  'Seq2Seq Baseline',   CFG['epochs'], train_loader, val_loader)

# Save baseline weights
torch.save(baseline_clstm.state_dict(),   OUT_DIR / 'baseline_convlstm.pt')
torch.save(baseline_seq2seq.state_dict(), OUT_DIR / 'baseline_seq2seq.pt')
print('\n✓ Baseline models saved.')

In [ ]:
# ── 5.4 Evaluate all models on test set (Req 4.4, 4.5) ───────────────────────
def evaluate_model(model, loader, name):
    model.eval()
    all_ssim, all_mae, all_rmse = [], [], []
    with torch.no_grad():
        for cond, real in loader:
            cond, real = cond.to(DEVICE), real.to(DEVICE)
            pred = model(cond)
            pred_np = np.clip(pred.cpu().numpy(), 0, 1)
            real_np = np.clip(real.cpu().numpy(), 0, 1)

            for b in range(pred_np.shape[0]):
                p = pred_np[b].transpose(1, 2, 0)
                r = real_np[b].transpose(1, 2, 0)
                all_ssim.append(ssim_fn(p, r, data_range=1.0, channel_axis=-1))
                all_mae.append(np.mean(np.abs(p - r)))
                all_rmse.append(np.sqrt(np.mean((p - r)**2)))

    results = dict(
        Model=name,
        SSIM =round(float(np.mean(all_ssim)),  4),
        MAE  =round(float(np.mean(all_mae)),   4),
        RMSE =round(float(np.mean(all_rmse)),  4),
    )
    return results


results_gan     = evaluate_model(netG,              test_loader, 'ST-CGAN')
results_clstm   = evaluate_model(baseline_clstm,   test_loader, 'ConvLSTM Baseline')
results_seq2seq = evaluate_model(baseline_seq2seq,  test_loader, 'Seq2Seq Baseline')

comparison_df = pd.DataFrame([results_gan, results_clstm, results_seq2seq])
comparison_df.set_index('Model', inplace=True)

print('\n' + '='*55)
print('  COMPARISON TABLE — Test Set Metrics')
print('='*55)
print(comparison_df.to_string())
print('='*55)

comparison_df.to_csv(OUT_DIR / 'comparison_table.csv')
print('\n✓ Comparison table saved.')

---
## 6. Evaluation & Visualization

In [ ]:
# ── 6.1 FID (Req 5.3) ────────────────────────────────────────────────────────
def compute_fid_approx(model, loader, n_samples=200):
    """
    Approximates FID using pixel-space feature statistics
    (mean and covariance of flattened patches).
    A full Inception-based FID is infeasible for vegetation-index data.
    """
    model.eval()
    reals, fakes = [], []
    total = 0
    with torch.no_grad():
        for cond, real in loader:
            if total >= n_samples:
                break
            cond, real = cond.to(DEVICE), real.to(DEVICE)
            fake = model(cond)
            reals.append(real.cpu().numpy().reshape(real.size(0), -1))
            fakes.append(fake.cpu().numpy().reshape(fake.size(0), -1))
            total += real.size(0)

    reals = np.clip(np.concatenate(reals, 0)[:n_samples], 0, 1)
    fakes = np.clip(np.concatenate(fakes, 0)[:n_samples], 0, 1)

    mu_r, mu_f = reals.mean(0), fakes.mean(0)
    cov_r = np.cov(reals, rowvar=False)
    cov_f = np.cov(fakes, rowvar=False)

    diff = mu_r - mu_f
    # FID ≈ ||mu_r - mu_f||^2 + Tr(Sigma_r + Sigma_f - 2*sqrt(Sigma_r*Sigma_f))
    # Simplified: use trace approximation
    fid = float(np.dot(diff, diff) + np.trace(cov_r) + np.trace(cov_f)
                - 2 * np.trace(np.sqrt(np.abs(cov_r @ cov_f) + 1e-8)))
    return fid

fid_score = compute_fid_approx(netG, test_loader)
print(f'Approx. FID (pixel-space): {fid_score:.4f}')

In [ ]:
# ── 6.2 Qualitative examples — 5 side-by-side plots (Req 5.4) ─────────────────
netG.eval()
cmap_veg = plt.cm.RdYlGn

all_cond, all_real, all_fake = [], [], []
with torch.no_grad():
    for cond_b, real_b in test_loader:
        cond_b, real_b = cond_b.to(DEVICE), real_b.to(DEVICE)
        fake_b = netG(cond_b)
        all_cond.append(cond_b.cpu()); all_real.append(real_b.cpu()); all_fake.append(fake_b.cpu())
        if len(all_cond) * CFG['batch_size'] >= 5:
            break

all_cond = torch.cat(all_cond, 0)
all_real = torch.cat(all_real, 0)
all_fake = torch.cat(all_fake, 0)

fig, axes = plt.subplots(5, 3, figsize=(12, 20))
fig.suptitle('Qualitative Examples: Input Condition | Ground Truth | ST-CGAN Generated',
             fontsize=13, fontweight='bold')

for i in range(5):
    # Show first channel (NDVI) of last condition frame
    cond_img = np.clip(all_cond[i, -CFG['channels']].numpy(), 0, 1)   # last NDVI frame
    real_img = np.clip(all_real[i, 0].numpy(), 0, 1)                  # target NDVI
    fake_img = np.clip(all_fake[i, 0].numpy(), 0, 1)                  # generated NDVI

    for ax, img, ttl in zip(axes[i], [cond_img, real_img, fake_img],
                            ['Condition (last NDVI)', 'Ground Truth', 'ST-CGAN']):
        im = ax.imshow(img, cmap=cmap_veg, vmin=0, vmax=1)
        ax.set_title(f'Sample {i+1} — {ttl}', fontsize=9)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(OUT_DIR / 'qualitative_examples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Qualitative examples saved.')

In [ ]:
# ── 6.3 Time-series of mean NDVI across test set (Req 5.5) ───────────────────
def collect_mean_ndvi(model, loader):
    model.eval()
    means = []
    with torch.no_grad():
        for cond, real in loader:
            cond = cond.to(DEVICE)
            pred = model(cond).cpu().numpy()
            # NDVI is channel 0
            means.append(np.clip(pred[:, 0], 0, 1).mean())
    return means


def gt_mean_ndvi(loader):
    vals = []
    for _, real in loader:
        vals.append(np.clip(real.numpy()[:, 0], 0, 1).mean())
    return vals


gan_ndvi     = collect_mean_ndvi(netG,              test_loader)
clstm_ndvi   = collect_mean_ndvi(baseline_clstm,   test_loader)
seq2seq_ndvi = collect_mean_ndvi(baseline_seq2seq,  test_loader)
gt_ndvi      = gt_mean_ndvi(test_loader)

fig, ax = plt.subplots(figsize=(14, 4))
steps = range(len(gt_ndvi))
ax.plot(steps, gt_ndvi,      label='Ground Truth',      color='black',     linewidth=2)
ax.plot(steps, gan_ndvi,     label='ST-CGAN',           color='steelblue', linewidth=1.5, linestyle='--')
ax.plot(steps, clstm_ndvi,   label='ConvLSTM Baseline', color='tomato',    linewidth=1.5, linestyle=':')
ax.plot(steps, seq2seq_ndvi, label='Seq2Seq Baseline',  color='goldenrod', linewidth=1.5, linestyle='-.')
ax.set_xlabel('Batch step (test set)')
ax.set_ylabel('Mean NDVI')
ax.set_title('Mean NDVI Time-Series: Generated vs Ground Truth vs Baselines')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'ndvi_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()
print('NDVI time-series plot saved.')

In [ ]:
# ── 6.4 Spatial error map — mean absolute error across test set (Req 5.6) ─────
netG.eval()
error_accum = None
n_accum     = 0

with torch.no_grad():
    for cond, real in test_loader:
        cond, real = cond.to(DEVICE), real.to(DEVICE)
        fake = netG(cond)
        err  = torch.abs(fake - real).mean(dim=1, keepdim=False)  # (B, P, P)
        if error_accum is None:
            error_accum = err.sum(0).cpu().numpy()
        else:
            error_accum += err.sum(0).cpu().numpy()
        n_accum += err.size(0)

spatial_mae = error_accum / max(n_accum, 1)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(spatial_mae, cmap='hot_r', vmin=0)
plt.colorbar(im, ax=ax, label='Mean Absolute Error')
ax.set_title('Spatial Error Map — Mean Absolute Pixel Error (ST-CGAN, Test Set)')
ax.axis('off')
plt.tight_layout()
plt.savefig(OUT_DIR / 'spatial_error_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Spatial error map saved.')

In [ ]:
# ── 6.5 Print final metrics summary ───────────────────────────────────────────
print('\n' + '='*55)
print('  FINAL EVALUATION SUMMARY')
print('='*55)
print(comparison_df.to_string())
print(f'\n  Approx. FID  (ST-CGAN):  {fid_score:.4f}')
print('='*55)

---
## 7. Uncertainty Estimation

In [ ]:
# ── 7.1 Monte Carlo dropout inference (Req 6.1, 6.2) ─────────────────────────
def mc_dropout_predict(model, x, n_samples=30):
    """
    Keep dropout ACTIVE during inference.
    Returns:
        mean_pred : (B, OUT_CH, P, P)
        std_pred  : (B, P, P)  — pixel-wise std across channels & samples
    """
    model.train()   # activates dropout
    preds = []
    with torch.no_grad():
        for _ in range(n_samples):
            preds.append(model(x).cpu().numpy())
    model.eval()

    preds = np.stack(preds, axis=0)           # (n_samples, B, C, P, P)
    mean_pred = preds.mean(axis=0)            # (B, C, P, P)
    std_pred  = preds.std(axis=0).mean(axis=1) # (B, P, P)
    return mean_pred, std_pred


# Collect predictions on the first batch of the test
test_iter = iter(test_loader)
cond_mc, real_mc = next(test_iter)
cond_mc = cond_mc.to(DEVICE)

mc_mean, mc_std = mc_dropout_predict(netG, cond_mc, n_samples=CFG['mc_samples'])
print(f'MC dropout: {CFG["mc_samples"]} samples collected.')

# Collect ALL test patches for aggregate statistics
all_mc_stds = [mc_std]
for cond_b, _ in test_loader:
    _, std_b = mc_dropout_predict(netG, cond_b.to(DEVICE), n_samples=CFG['mc_samples'])
    all_mc_stds.append(std_b)

all_mc_stds_flat = np.concatenate([s.flatten() for s in all_mc_stds])
mean_unc   = float(np.mean(all_mc_stds_flat))
p95_unc    = float(np.percentile(all_mc_stds_flat, 95))

print(f'\nUncertainty across all test patches:')
print(f'  Mean uncertainty  : {mean_unc:.6f}')
print(f'  95th-pct uncertainty: {p95_unc:.6f}')

In [ ]:
# ── 7.2 Visualise uncertainty maps for 3 patches (Req 6.3) ───────────────────
N_VIZ = min(3, cond_mc.size(0))
fig, axes = plt.subplots(N_VIZ, 3, figsize=(13, 4 * N_VIZ))
if N_VIZ == 1:
    axes = axes[np.newaxis, :]

fig.suptitle('Uncertainty Estimation — Predicted NDVI | Ground Truth | Uncertainty Map',
             fontsize=12, fontweight='bold')

real_mc_np = real_mc[:N_VIZ, 0].numpy()

for i in range(N_VIZ):
    pred_img = np.clip(mc_mean[i, 0], 0, 1)
    real_img = np.clip(real_mc_np[i],   0, 1)
    unc_img  = mc_std[i]

    for ax, img, ttl, cmap in zip(
        axes[i],
        [pred_img, real_img, unc_img],
        ['MC-Mean Prediction', 'Ground Truth', 'Uncertainty (std)'],
        [cmap_veg, cmap_veg, 'plasma']
    ):
        im = ax.imshow(img, cmap=cmap, vmin=0, vmax=img.max() if ttl.startswith('Unc') else 1)
        ax.set_title(f'Patch {i+1} — {ttl}', fontsize=9)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(OUT_DIR / 'uncertainty_maps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Uncertainty maps saved.')

In [ ]:
# ── 7.3 Uncertainty distribution histogram ────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(all_mc_stds_flat, bins=80, color='mediumpurple', edgecolor='white', linewidth=0.4)
ax.axvline(mean_unc, color='black', linestyle='--', linewidth=1.5, label=f'Mean = {mean_unc:.4f}')
ax.axvline(p95_unc,  color='tomato', linestyle=':',  linewidth=1.5, label=f'95th pct = {p95_unc:.4f}')
ax.set_xlabel('Pixel-wise Std Dev (Uncertainty)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Prediction Uncertainty (MC Dropout, 30 samples)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'uncertainty_histogram.png', dpi=150, bbox_inches='tight')
plt.show()
print('Uncertainty histogram saved.')

print(f'\n✅ All outputs written to {OUT_DIR}')
print('\nSaved files:')
for f in sorted(OUT_DIR.iterdir()):
    print(f'  {f.name}')

---

## Summary

| Section | Status |
|---|---|
| Data Loading & Preprocessing | ✅ Req 1.1–1.7 |
| ST-CGAN Architecture | ✅ Req 2.1–2.7 |
| Training (200 epochs, checkpoints, CSV log) | ✅ Req 3.1–3.8 |
| Baseline Comparison (ConvLSTM, Seq2Seq) | ✅ Req 4.1–4.5 |
| Evaluation & Visualization | ✅ Req 5.1–5.6 |
| Uncertainty Estimation (MC dropout × 30) | ✅ Req 6.1–6.4 |
| Notebook Structure & Kaggle Compatibility | ✅ Req 7.1–7.7 |